## Example 1: Basic Import and Inspection\n
\n
Basic import test and inspection of gpytorch package structure.\n

In [ ]:
# Example 1: Basic Gaussian Process Regression with GPyTorch
import torch
import gpytorch
import numpy as np

print("Basic Gaussian Process regression with GPyTorch:")
print("=" * 49)

# Generate synthetic training data
np.random.seed(42)
train_x = torch.linspace(0, 1, 20)
train_y = torch.sin(train_x * (2 * np.pi)) + torch.randn(train_x.size()) * 0.1

print(f"1. Training data generation:")
print(f"   X shape: {train_x.shape}")
print(f"   y shape: {train_y.shape}")
print(f"   X range: [{train_x.min():.2f}, {train_x.max():.2f}]")
print(f"   y range: [{train_y.min():.2f}, {train_y.max():.2f}]")

# Define a simple GP model
class ExactGPModel(gpytorch.models.ExactGP):
    def __init__(self, train_x, train_y, likelihood):
        super(ExactGPModel, self).__init__(train_x, train_y, likelihood)
        self.mean_module = gpytorch.means.ConstantMean()
        self.covar_module = gpytorch.kernels.ScaleKernel(gpytorch.kernels.RBFKernel())
    
    def forward(self, x):
        mean_x = self.mean_module(x)
        covar_x = self.covar_module(x)
        return gpytorch.distributions.MultivariateNormal(mean_x, covar_x)

print(f"\n2. Model initialization:")
likelihood = gpytorch.likelihoods.GaussianLikelihood()
model = ExactGPModel(train_x, train_y, likelihood)

print(f"   Model type: {type(model).__name__}")
print(f"   Likelihood: {type(likelihood).__name__}")
print(f"   Mean module: {type(model.mean_module).__name__}")
print(f"   Covar module: {type(model.covar_module).__name__}")

# Training mode
model.train()
likelihood.train()

print(f"\n3. Model training:")
optimizer = torch.optim.Adam(model.parameters(), lr=0.1)
mll = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, model)

# Training loop
training_iter = 50
for i in range(training_iter):
    optimizer.zero_grad()
    output = model(train_x)
    loss = -mll(output, train_y)
    loss.backward()
    optimizer.step()
    
    if (i + 1) % 10 == 0:
        print(f"   Iteration {i+1}/{training_iter} - Loss: {loss.item():.3f}")

# Evaluation mode
model.eval()
likelihood.eval()

print(f"\n4. Making predictions:")
with torch.no_grad(), gpytorch.settings.fast_pred_var():
    test_x = torch.linspace(0, 1, 51)
    predictions = likelihood(model(test_x))
    mean = predictions.mean
    variance = predictions.variance

print(f"   Test points: {len(test_x)}")
print(f"   Prediction range: [{mean.min():.2f}, {mean.max():.2f}]")
print(f"   Mean variance: {variance.mean():.4f}")

print("\nBasic Gaussian Process regression completed successfully!")

## Example 2: Core Functionality Test\n
\n
Test core functionality of gpytorch package.\n

In [ ]:
# Example 2: Multi-output Gaussian Process
import torch
import gpytorch
import numpy as np

print("Multi-output Gaussian Process:")
print("=" * 29)

# Generate multi-output training data
np.random.seed(123)
n_train = 30
train_x = torch.rand(n_train)
train_y = torch.stack([
    torch.sin(train_x * 2 * np.pi),
    torch.cos(train_x * 2 * np.pi)
], -1)

print(f"1. Multi-output data:")
print(f"   Input shape: {train_x.shape}")
print(f"   Output shape: {train_y.shape}")
print(f"   Number of tasks: {train_y.shape[-1]}")

# Multi-task GP Model
class MultitaskGPModel(gpytorch.models.ExactGP):
    def __init__(self, train_x, train_y, likelihood):
        super(MultitaskGPModel, self).__init__(train_x, train_y, likelihood)
        self.mean_module = gpytorch.means.MultitaskMean(
            gpytorch.means.ConstantMean(), num_tasks=2
        )
        self.covar_module = gpytorch.kernels.MultitaskKernel(
            gpytorch.kernels.RBFKernel(), num_tasks=2, rank=1
        )
    
    def forward(self, x):
        mean_x = self.mean_module(x)
        covar_x = self.covar_module(x)
        return gpytorch.distributions.MultitaskMultivariateNormal(mean_x, covar_x)

print(f"\n2. Multi-task model setup:")
likelihood = gpytorch.likelihoods.MultitaskGaussianLikelihood(num_tasks=2)
model = MultitaskGPModel(train_x, train_y, likelihood)

print(f"   Model: {type(model).__name__}")
print(f"   Likelihood: {type(likelihood).__name__}")
print(f"   Number of tasks: 2")

# Train the model
model.train()
likelihood.train()

print(f"\n3. Training multi-task GP:")
optimizer = torch.optim.Adam(model.parameters(), lr=0.1)
mll = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, model)

for i in range(50):
    optimizer.zero_grad()
    output = model(train_x)
    loss = -mll(output, train_y)
    loss.backward()
    optimizer.step()
    
    if (i + 1) % 10 == 0:
        print(f"   Iteration {i+1}/50 - Loss: {loss.item():.3f}")

# Make predictions
model.eval()
likelihood.eval()

print(f"\n4. Multi-task predictions:")
with torch.no_grad():
    test_x = torch.linspace(0, 1, 50)
    predictions = likelihood(model(test_x))
    mean = predictions.mean

print(f"   Test points: {len(test_x)}")
print(f"   Output dimensions: {mean.shape}")
print(f"   Task 1 prediction range: [{mean[:, 0].min():.2f}, {mean[:, 0].max():.2f}]")
print(f"   Task 2 prediction range: [{mean[:, 1].min():.2f}, {mean[:, 1].max():.2f}]")

print("\nMulti-output Gaussian Process completed successfully!")

## Example 3: Error Handling and Edge Cases\n
\n
Test error handling and edge cases for gpytorch.\n

In [ ]:
# Example 3: Deep Kernel Learning
import torch
import gpytorch
import numpy as np

print("Deep Kernel Learning:")
print("=" * 20)

# Generate non-linear data
np.random.seed(456)
n_train = 100
train_x = torch.rand(n_train, 2) * 2 - 1  # 2D input in [-1, 1]
train_y = (train_x[:, 0] ** 2 + train_x[:, 1] ** 2).sin() + torch.randn(n_train) * 0.1

print(f"1. Non-linear data:")
print(f"   Input shape: {train_x.shape}")
print(f"   Output shape: {train_y.shape}")
print(f"   Target range: [{train_y.min():.2f}, {train_y.max():.2f}]")

# Feature extractor network
class LargeFeatureExtractor(torch.nn.Sequential):
    def __init__(self, input_dim):
        super(LargeFeatureExtractor, self).__init__()
        self.add_module('linear1', torch.nn.Linear(input_dim, 16))
        self.add_module('relu1', torch.nn.ReLU())
        self.add_module('linear2', torch.nn.Linear(16, 8))
        self.add_module('relu2', torch.nn.ReLU())

# Deep GP Model
class DKLModel(gpytorch.models.ExactGP):
    def __init__(self, train_x, train_y, likelihood):
        super(DKLModel, self).__init__(train_x, train_y, likelihood)
        self.mean_module = gpytorch.means.ConstantMean()
        self.covar_module = gpytorch.kernels.ScaleKernel(gpytorch.kernels.RBFKernel())
        self.feature_extractor = LargeFeatureExtractor(train_x.size(-1))
    
    def forward(self, x):
        projected_x = self.feature_extractor(x)
        mean_x = self.mean_module(projected_x)
        covar_x = self.covar_module(projected_x)
        return gpytorch.distributions.MultivariateNormal(mean_x, covar_x)

print(f"\n2. Deep kernel learning model:")
likelihood = gpytorch.likelihoods.GaussianLikelihood()
model = DKLModel(train_x, train_y, likelihood)

print(f"   Model: {type(model).__name__}")
print(f"   Feature extractor layers: 4")
print(f"   GP kernel: RBF")

# Train the model
model.train()
likelihood.train()

print(f"\n3. Training deep GP:")
optimizer = torch.optim.Adam([
    {'params': model.feature_extractor.parameters()},
    {'params': model.covar_module.parameters()},
    {'params': model.mean_module.parameters()},
    {'params': likelihood.parameters()},
], lr=0.01)

mll = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, model)

for i in range(50):
    optimizer.zero_grad()
    output = model(train_x)
    loss = -mll(output, train_y)
    loss.backward()
    optimizer.step()
    
    if (i + 1) % 10 == 0:
        print(f"   Iteration {i+1}/50 - Loss: {loss.item():.3f}")

# Evaluate
model.eval()
likelihood.eval()

print(f"\n4. Evaluation:")
with torch.no_grad():
    test_x = torch.rand(20, 2) * 2 - 1
    predictions = likelihood(model(test_x))
    mean = predictions.mean
    variance = predictions.variance

print(f"   Test samples: {len(test_x)}")
print(f"   Prediction range: [{mean.min():.2f}, {mean.max():.2f}]")
print(f"   Mean uncertainty: {variance.mean().sqrt():.4f}")

print("\nDeep Kernel Learning completed successfully!")

## Example 4: Integration and Usage Patterns\n
\n
Test integration patterns and common usage of gpytorch.\n

In [ ]:
# Example 4: Variational GP for Large Datasets
import torch
import gpytorch
import numpy as np

print("Variational GP for large datasets:")
print("=" * 34)

# Generate large dataset
np.random.seed(789)
n_train = 1000
train_x = torch.rand(n_train) * 10
train_y = torch.sin(train_x) + torch.randn(n_train) * 0.2

print(f"1. Large dataset:")
print(f"   Training samples: {n_train}")
print(f"   Input range: [{train_x.min():.2f}, {train_x.max():.2f}]")

# Variational GP Model with inducing points
class VariationalGPModel(gpytorch.models.ApproximateGP):
    def __init__(self, inducing_points):
        variational_distribution = gpytorch.variational.CholeskyVariationalDistribution(
            inducing_points.size(0)
        )
        variational_strategy = gpytorch.variational.VariationalStrategy(
            self, inducing_points, variational_distribution, learn_inducing_locations=True
        )
        super(VariationalGPModel, self).__init__(variational_strategy)
        self.mean_module = gpytorch.means.ConstantMean()
        self.covar_module = gpytorch.kernels.ScaleKernel(gpytorch.kernels.RBFKernel())
    
    def forward(self, x):
        mean_x = self.mean_module(x)
        covar_x = self.covar_module(x)
        return gpytorch.distributions.MultivariateNormal(mean_x, covar_x)

print(f"\n2. Variational GP setup:")
n_inducing = 100
inducing_points = torch.linspace(0, 10, n_inducing)
model = VariationalGPModel(inducing_points)
likelihood = gpytorch.likelihoods.GaussianLikelihood()

print(f"   Inducing points: {n_inducing}")
print(f"   Model: {type(model).__name__}")
print(f"   Variational strategy: VariationalStrategy")

# Training
model.train()
likelihood.train()

print(f"\n3. Training variational GP:")
optimizer = torch.optim.Adam([
    {'params': model.parameters()},
    {'params': likelihood.parameters()},
], lr=0.01)

mll = gpytorch.mlls.VariationalELBO(likelihood, model, num_data=train_y.size(0))

batch_size = 100
for i in range(50):
    # Mini-batch training
    indices = torch.randperm(n_train)[:batch_size]
    x_batch = train_x[indices]
    y_batch = train_y[indices]
    
    optimizer.zero_grad()
    output = model(x_batch)
    loss = -mll(output, y_batch)
    loss.backward()
    optimizer.step()
    
    if (i + 1) % 10 == 0:
        print(f"   Iteration {i+1}/50 - Loss: {loss.item():.3f}")

# Predictions
model.eval()
likelihood.eval()

print(f"\n4. Predictions with inducing points:")
with torch.no_grad():
    test_x = torch.linspace(0, 10, 200)
    predictions = likelihood(model(test_x))
    mean = predictions.mean
    variance = predictions.variance

print(f"   Test points: {len(test_x)}")
print(f"   Prediction range: [{mean.min():.2f}, {mean.max():.2f}]")
print(f"   Mean uncertainty: {variance.mean().sqrt():.4f}")

print("\nVariational GP for large datasets completed successfully!")

## Example 5: Comprehensive Feature Test\n
\n
Comprehensive test of gpytorch features and capabilities.\n

In [ ]:
# Example 5: Spectral Mixture Kernel
import torch
import gpytorch
import numpy as np

print("Spectral Mixture Kernel GP:")
print("=" * 27)

# Generate periodic data
np.random.seed(101)
n_train = 100
train_x = torch.linspace(0, 4 * np.pi, n_train)
train_y = torch.sin(train_x) + 0.5 * torch.sin(3 * train_x) + torch.randn(n_train) * 0.1

print(f"1. Periodic data:")
print(f"   Training samples: {n_train}")
print(f"   Input range: [0, {4 * np.pi:.2f}]")
print(f"   Multiple frequency components")

# GP with Spectral Mixture Kernel
class SpectralMixtureGPModel(gpytorch.models.ExactGP):
    def __init__(self, train_x, train_y, likelihood):
        super(SpectralMixtureGPModel, self).__init__(train_x, train_y, likelihood)
        self.mean_module = gpytorch.means.ConstantMean()
        self.covar_module = gpytorch.kernels.SpectralMixtureKernel(num_mixtures=4)
        self.covar_module.initialize_from_data(train_x, train_y)
    
    def forward(self, x):
        mean_x = self.mean_module(x)
        covar_x = self.covar_module(x)
        return gpytorch.distributions.MultivariateNormal(mean_x, covar_x)

print(f"\n2. Spectral Mixture Kernel setup:")
likelihood = gpytorch.likelihoods.GaussianLikelihood()
model = SpectralMixtureGPModel(train_x, train_y, likelihood)

print(f"   Model: {type(model).__name__}")
print(f"   Kernel: SpectralMixtureKernel")
print(f"   Number of mixture components: 4")
print(f"   Initialized from data")

# Training
model.train()
likelihood.train()

print(f"\n3. Training spectral mixture GP:")
optimizer = torch.optim.Adam(model.parameters(), lr=0.1)
mll = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, model)

for i in range(100):
    optimizer.zero_grad()
    output = model(train_x)
    loss = -mll(output, train_y)
    loss.backward()
    optimizer.step()
    
    if (i + 1) % 20 == 0:
        print(f"   Iteration {i+1}/100 - Loss: {loss.item():.3f}")

# Predictions
model.eval()
likelihood.eval()

print(f"\n4. Capturing periodic patterns:")
with torch.no_grad():
    # Extrapolate beyond training range
    test_x = torch.linspace(0, 6 * np.pi, 200)
    predictions = likelihood(model(test_x))
    mean = predictions.mean
    lower, upper = predictions.confidence_region()

print(f"   Test range: [0, {6 * np.pi:.2f}] (extrapolation)")
print(f"   Prediction range: [{mean.min():.2f}, {mean.max():.2f}]")
print(f"   Confidence region width: {(upper - lower).mean():.4f}")

# Analyze learned frequencies
print(f"\n5. Learned spectral characteristics:")
mixture_means = model.covar_module.mixture_means.detach()
mixture_scales = model.covar_module.mixture_scales.detach()

print(f"   Mixture component frequencies:")
for i in range(min(4, len(mixture_means))):
    freq = mixture_means[i].item()
    scale = mixture_scales[i].item()
    print(f"     Component {i+1}: freq={freq:.3f}, scale={scale:.3f}")

print("\nSpectral Mixture Kernel GP completed successfully!")